## VLM Shellgame

- VLM is glm-5.3-flash
- Videos: A,B,C labeled on the table with 3 inverted cups. Each video starts by showing where the puck is then covers it with a cup.  
then cups are moved around in shell game style.
- evaluated on single videos of shell game in POV.
- 3 types of videos: slow, medium, fast to see temporal resoning of the model
- For each video model picks from A,B,C


In [1]:
## conda env vgd has openai
import base64
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import numpy as np
from tqdm import tqdm

import cv2

In [2]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

# Preparing main index_df with vqa videos

In [15]:
## index_df
index_df = pd.read_csv('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/index.csv')
print(index_df.shape)

index_df = index_df[index_df['session'].str.startswith('shellgame')].reset_index(drop=True)
print(index_df.shape)

index_df['cam'] = 'cam0'
index_df = index_df.rename(columns={'id':'video_id'})

print(index_df.shape)
index_df.head(2)

(228, 4)
(32, 4)
(32, 5)


,video_id,template,placeholders,session,cam
0,261117-155127,C,PUCK,shellgame-medium,cam0
1,261117-155207,B,PUCK,shellgame-medium,cam0


In [16]:
labels_list = index_df.template.unique().tolist()
labels_list.sort()

index_df.template.value_counts()

template
C    11
A    11
B    10
Name: count, dtype: int64

In [17]:
index_df.session.value_counts()

session
shellgame-slow      12
shellgame-medium    10
shellgame-fast      10
Name: count, dtype: int64

### select videos

In [18]:
for i, row in index_df.iterrows():
    print(f"'{row['video_id']}', # --> {row['template']:5} -- {row['session']} ")


'261117-155127', # --> C     -- shellgame-medium 
'261117-155207', # --> B     -- shellgame-medium 
'261117-155252', # --> A     -- shellgame-medium 
'261117-155426', # --> C     -- shellgame-medium 
'261117-155503', # --> C     -- shellgame-medium 
'261117-155543', # --> B     -- shellgame-fast 
'261117-155621', # --> C     -- shellgame-fast 
'261117-155722', # --> B     -- shellgame-fast 
'261117-160815', # --> A     -- shellgame-fast 
'261117-160858', # --> A     -- shellgame-fast 
'261117-160936', # --> B     -- shellgame-fast 
'261117-161017', # --> C     -- shellgame-fast 
'261117-161053', # --> B     -- shellgame-fast 
'261117-161121', # --> C     -- shellgame-fast 
'261117-161150', # --> C     -- shellgame-fast 
'261117-161257', # --> B     -- shellgame-medium 
'261117-161345', # --> A     -- shellgame-medium 
'261117-161420', # --> A     -- shellgame-medium 
'261117-161455', # --> B     -- shellgame-medium 
'261117-161544', # --> A     -- shellgame-medium 
'261117-161811', # -

In [19]:
vids_list = [
    '261117-155127', # --> C     -- shellgame-medium 
    '261117-155207', # --> B     -- shellgame-medium 
    '261117-155252', # --> A     -- shellgame-medium 
    '261117-155426', # --> C     -- shellgame-medium 
    '261117-155503', # --> C     -- shellgame-medium 
    '261117-155543', # --> B     -- shellgame-fast 
    '261117-155621', # --> C     -- shellgame-fast 
    '261117-155722', # --> B     -- shellgame-fast 
    '261117-160815', # --> A     -- shellgame-fast 
    '261117-160858', # --> A     -- shellgame-fast 
    '261117-160936', # --> B     -- shellgame-fast 
    '261117-161017', # --> C     -- shellgame-fast 
    '261117-161053', # --> B     -- shellgame-fast 
    '261117-161121', # --> C     -- shellgame-fast 
    '261117-161150', # --> C     -- shellgame-fast 
    '261117-161257', # --> B     -- shellgame-medium 
    '261117-161345', # --> A     -- shellgame-medium 
    '261117-161420', # --> A     -- shellgame-medium 
    '261117-161455', # --> B     -- shellgame-medium 
    '261117-161544', # --> A     -- shellgame-medium 
    '261117-161811', # --> C     -- shellgame-slow 
    '261117-161859', # --> A     -- shellgame-slow 
    '261117-161936', # --> A     -- shellgame-slow 
    '261117-162009', # --> C     -- shellgame-slow 
    '261117-162259', # --> A     -- shellgame-slow 
    '261117-162319', # --> C     -- shellgame-slow 
    '261117-162343', # --> B     -- shellgame-slow 
    '261117-162402', # --> B     -- shellgame-slow 
    '261117-162434', # --> A     -- shellgame-slow 
    '261117-162457', # --> A     -- shellgame-slow 
    '261117-162515', # --> B     -- shellgame-slow 
    '261117-162538', # --> C     -- shellgame-slow 

]
print(len(vids_list))

32


In [20]:
print(index_df.shape)
index_df = index_df[index_df.video_id.isin(vids_list)]
print(index_df.shape)

(32, 5)
(32, 5)


In [13]:
import shutil
# ------------- chunk to copy videos ------------- #
source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

# 2. Clear destination directory if it exists, then recreate it
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)
os.makedirs(dest_dir, exist_ok=True)

# 3. Get unique video IDs and copy matching files
unique_video_ids = vids_list

for filename in os.listdir(source_dir):
    if any(video_id in filename for video_id in unique_video_ids):
        shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
# ------------------------------------------------ #

# VLM Evaluation 

### Compress Videos. 9second videos too long and high def. glm-5.3-Flash can handle <10MB vids and mine are well over 15mb per single view

## compress to lower res and store as mp4

In [46]:
import os
import subprocess

raw_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw"
compressed_dir = "data/compressed_videos"

os.makedirs(compressed_dir, exist_ok=True)

for vid in vids_list:
    # Strip extension if vids_list already includes '.avi'
    stem = os.path.splitext(vid)[0]
    
    input_path = os.path.join(raw_dir, f"{stem}_cam0.avi")       # Change to f"{stem}_cam0.avi" if suffix is still used
    output_path = os.path.join(compressed_dir, f"{stem}_cam0.mp4")

    if os.path.exists(output_path):
        print(f"[-] Skipping (already exists): {stem}_cam0.mp4")
        continue

    if not os.path.exists(input_path):
        print(f"[!] Warning (missing input): {input_path}")
        continue

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-loglevel", "error",
            "-i", input_path,
            "-vf", "fps=20,scale=-2:480p",
            "-c:v", "libx264",
            "-crf", "23",
            "-an",
            output_path,
        ],
        check=True,
    )

    orig_mb = os.path.getsize(input_path) / (1024 * 1024)
    new_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"[*] Converted {stem}: {orig_mb:.2f} MB -> {new_mb:.2f} MB")

[-] Skipping (already exists): 261117-155127_cam0.mp4
[-] Skipping (already exists): 261117-155207_cam0.mp4
[*] Converted 261117-155252: 15.34 MB -> 2.31 MB
[-] Skipping (already exists): 261117-155426_cam0.mp4
[-] Skipping (already exists): 261117-155503_cam0.mp4
[*] Converted 261117-155543: 10.72 MB -> 1.71 MB
[*] Converted 261117-155621: 15.05 MB -> 2.45 MB
[*] Converted 261117-155722: 14.99 MB -> 2.41 MB
[*] Converted 261117-160815: 11.08 MB -> 1.75 MB
[*] Converted 261117-160858: 15.47 MB -> 2.41 MB
[*] Converted 261117-160936: 12.56 MB -> 1.97 MB
[*] Converted 261117-161017: 15.04 MB -> 2.35 MB
[*] Converted 261117-161053: 11.51 MB -> 1.75 MB
[*] Converted 261117-161121: 11.00 MB -> 1.67 MB
[*] Converted 261117-161150: 17.47 MB -> 2.64 MB
[*] Converted 261117-161257: 14.33 MB -> 1.99 MB
[*] Converted 261117-161345: 15.82 MB -> 2.31 MB
[*] Converted 261117-161420: 14.69 MB -> 2.13 MB
[*] Converted 261117-161455: 15.23 MB -> 2.09 MB
[*] Converted 261117-161544: 16.18 MB -> 2.29 MB


In [ ]:
## 6mins total to compress all 32 videos to 480p 20fps mp4 format.

### Run one video

In [38]:
# Load environment variables from .env
load_dotenv()

client = OpenAI(
    api_key=os.getenv("GWDG_OPENAI_API_KEY"), 
    base_url="https://chat-ai.academiccloud.de/v1"
    )
client

In [39]:
# Encode combined video to base64
video_name = vids_list[0]

video_file_path = f'data/compressed_videos/{video_name}_cam0.mp4'

with open(video_file_path, "rb") as f:
    b64_video = base64.b64encode(f.read()).decode("utf-8")
    b64_size_mb = len(b64_video) / (1024 ** 2)
    print(f"Base64 payload size: {b64_size_mb:.2f} MB")

Base64 payload size: 3.03 MB


In [ ]:
system_prompt = (
    "You are an expert visual object-tracking and spatial-temporal reasoning model. "
    "You are provided with a single bird's-eye view video of a table with three clearly marked positions: A, B, and C, each having an inverted cup. "
    "At the beginning of the video, a puck is revealed under one of the cups, after which the cups are shuffled. "
    "Track the target cup continuously across all movements to determine its final position."
)

user_prompt = f"""
Analyze this bird's-eye view video of a shell game and determine which position (A, B, or C) currently hides the puck at the end of the video.

Valid Positions: {{"A", "B", "C"}}

Instructions:
1. Identify the starting position (A, B, or C) where the puck is initially revealed under a cup at the start of the video.
2. Track the cup with the puck chronologically through every move or swap from start to finish, noting how the target cup's position changes at each step.
3. State your final predicted position ("A", "B", or "C") where the puck is located at the end of the video.
4. Provide a confidence score as a float between 0.0 and 1.0 denoting your certainty in the final predicted position (where 1.0 is completely certain and 0.33 is random guessing).

Output strictly in JSON matching this schema:|
{{
  "initial_position": "<A, B, or C>",
  "swap_sequence": "A -> C -> A -> B",
  "predicted_position": "<A, B, or C>",
  "confidence": 0.0
}}
"""

print(user_prompt)


Analyze this bird's-eye view video of a shell game and determine which position (A, B, or C) currently hides the puck at the end of the video.

Valid Positions: {"A", "B", "C"}

Instructions:
1. Identify the starting position (A, B, or C) where the puck is initially revealed under a cup at the start of the video.
2. Track the cup with the puck chronologically through every move or swap from start to finish, noting how the target cup's position changes at each step.
3. State your final predicted position ("A", "B", or "C") where the puck is located at the end of the video.
4. Provide a confidence score as a float between 0.0 and 1.0 denoting your certainty in the final predicted position (where 1.0 is completely certain and 0.33 is random guessing).

Output strictly in JSON matching this schema:|
{
  "initial_position": "<A, B, or C>",
  "swap_sequence": "A -> C -> A -> B",  # Example format showing the chronological sequence of cup positions
  "predicted_position": "<A, B, or C>",
  "

In [41]:
start_time = time.time()

response = client.chat.completions.create(
    model="glm-5.3-flash",
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_prompt},
                {
                    "type": "video_url",
                    "video_url": {"url": f"data:video/mp4;base64,{b64_video}"},
                },
            ],
        },
    ],
    response_format={"type": "json_object"},
    temperature=0.1,
    # timeout=120  # Set a timeout of 120 seconds
)

elapsed_time = round(time.time() - start_time, 2)

print(f"Elapsed time: {elapsed_time}s")

Elapsed time: 1343.85s


In [42]:
prediction_json = json.loads(response.choices[0].message.content)
prediction_json.keys()

dict_keys(['initial_position', 'swap_sequence', 'predicted_position', 'confidence'])

In [ ]:
# Extract JSON prediction content
prediction_json = json.loads(response.choices[0].message.content)

# Extract token & performance metadata
usage = response.usage

# 3. Update the dictionary to log both video file names
run_result = {
    "video_file": os.path.basename(video_file_path),
    "model": response.model,
    "elapsed_seconds": elapsed_time,
    "prompt_tokens": usage.prompt_tokens if usage else None,
    "completion_tokens": usage.completion_tokens if usage else None,
    "total_tokens": usage.total_tokens if usage else None,
    "initial_position": prediction_json.get("initial_position"),
    "predicted_position": prediction_json.get("predicted_position"),
    "confidence": float(prediction_json.get("confidence", 0.0)),
    "swap_sequence": prediction_json.get("swap_sequence"),
}

# Append result as a single JSON line
with open('data/vlm_eval_results/shellgame_predictions.jsonl', "a", encoding="utf-8") as f:
    f.write(json.dumps(run_result) + "\n")

print(json.dumps(run_result, indent=2))

{
  "video_file": "261117-155127_cam0.mp4",
  "model": "glm-5.3-flash",
  "elapsed_seconds": 1343.85,
  "prompt_tokens": 9975,
  "completion_tokens": 1072,
  "total_tokens": 11047,
  "initial_position": "A",
  "predicted_position": "A",
  "confidence": 1.0,
  "swap_sequence": "A -> B -> B -> A"
}


## Main Experiment: For loop over many videos

In [70]:
# Load environment variables from .env
load_dotenv()

client = OpenAI(
    api_key=os.getenv("GWDG_OPENAI_API_KEY"), 
    base_url="https://chat-ai.academiccloud.de/v1"
)

In [73]:
len(vids_list[1:])

26

In [ ]:
from tqdm import tqdm

videos_dir = 'data/compressed_videos/'

pbar = tqdm(vids_list[1:], desc="Processing Videos")

for video_name in pbar:
    # Make file path and load video in base64 format
    video_file_path = videos_dir + video_name + '_cam0.mp4'
    with open(video_file_path, "rb") as f:
        b64_video = base64.b64encode(f.read()).decode("utf-8")

    # system_prompts and user_prompt exactly same for all videos.
    
    # API call
    start_time = time.time()

    response = client.chat.completions.create(
        model="glm-5.3-flash",
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_prompt},
                    {
                        "type": "video_url",
                        "video_url": {"url": f"data:video/mp4;base64,{b64_video}"},
                    },
                ],
            },
        ],
        response_format={"type": "json_object"},
        temperature=0.1,
    )

    elapsed_time = round(time.time() - start_time, 2)

    # Extract JSON prediction content
    prediction_json = json.loads(response.choices[0].message.content)

    # Extract token & performance metadata
    usage = response.usage

    # 3. Update the dictionary to log both video file names
    run_result = {
        "video_file": os.path.basename(video_file_path),
        "model": response.model,
        "elapsed_seconds": elapsed_time,
        "prompt_tokens": usage.prompt_tokens if usage else None,
        "completion_tokens": usage.completion_tokens if usage else None,
        "total_tokens": usage.total_tokens if usage else None,
        "initial_position": prediction_json.get("initial_position"),
        "predicted_position": prediction_json.get("predicted_position"),
        "confidence": float(prediction_json.get("confidence", 0.0)),
        "swap_sequence": prediction_json.get("swap_sequence"),
    }

    # Append result as a single JSON line
    with open('data/vlm_eval_results/shellgame_predictions.jsonl', "a", encoding="utf-8") as f:
        f.write(json.dumps(run_result) + "\n")


Processing Videos: 100%|██████████| 26/26 [07:53<00:00, 18.20s/it]


# Loading and analyzing the jsonl outputs of the VLM
## Main Experiment: shellgame

In [ ]:
# 1. Load jsonl file directly into a DataFrame
df = pd.read_json("data/vlm_eval_results/action_predictions_shellgame.jsonl", lines=True)

print(df.shape)
print(df.head(10))

(27, 11)
                   video_file          model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens                                                predicted_class                                                                                                                                                         view_0_observation                                                                                                                                                                      view_1_observation                                                                                                                                                                                                                                                                   combined_reasoning                                                                                                                                                                                           

In [ ]:
index_df['video_file'] = index_df.video_id + '_cam0.avi'
print(index_df.shape)

df = pd.merge(df, index_df, how = 'left', on = ['video_file']) # LEFT JOIN
print(df.shape)
print(df.sample(3))

(27, 8)
(27, 18)
                    video_file          model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens                                         predicted_class                                                                                                                                       view_0_observation                                                                                                                                                            view_1_observation                                                                                                                                                                                                                               combined_reasoning                                                                                                                                                                                                                                                         

In [ ]:
df['correct_top1'] = df['predicted_position'] == df['template']

print(f'Examples: {df.shape[0]}')
print(f'correct: {df.correct_top1.sum():4}')
print(f'top1acc: {df.correct_top1.mean():4}')

Examples: 27
correct:   21
top1acc: 0.7777777777777778


In [ ]:
import textwrap
import pandas as pd

pd.set_option('display.float_format', '{:,.3f}'.format)

def analyze_df(df):
    print('=' * 20, 'DF Stats by Session', '=' * 20)

    stats = df.groupby(['session']).agg(
        videos=('video_file', 'count'),
        top1_acc=('correct_top1', 'mean'),
        avg_confidence=('confidence', 'mean'),
    )
    print(stats)
    print('\n')

    print('=' * 20, 'Individual Videos by session_special', '=' * 20)

    for session in df['session_special'].unique():
        print('\n' + '-' * 20, session, '-' * 20)

        for i, row in df[df['session_special'] == session].iterrows():
            # Include ground-truth column if present in your merged DataFrame
            summary_cols = [
                col for col in [
                    'video_id', 'session', 'initial_position',
                    'true_position', 'predicted_position', 'confidence', 'correct_top1'
                ] if col in df.columns
            ]
            print(row[summary_cols].to_frame().T.to_string(index=False))

            print('\nSwap Sequence: ', textwrap.fill(str(row['swap_sequence']), width=100))
            print('-' * 60)

In [81]:
analyze_df(df)

==================== DF Stats by Template ====================
                                                                   videos  top1acc
template                                                                          
Moving [something] across a surface until it falls down                 2    1.000
Moving [something] across a surface until it hits [something]           2    1.000
Picking [something] up and deforming it                                 2    1.000
Picking [something] up then laying it down on its side                  1    1.000
Picking [something] up then putting it down upright                     2    1.000
Picking [something] up while putting [something] down                   2    1.000
Rolling [something] across a surface until it falls down                1    1.000
Rolling [something] across a surface until it hits [something]          2    1.000
Rolling [something] on a surface and stop it with hand                  2    0.500
Tearing [something] and 

In [136]:
from random import shuffle

x = [i for i in range(10)]
print(x)
shuffle(x)
print(x)


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[6, 2, 3, 9, 5, 0, 4, 1, 7, 8]
